# GPU scattering, convergence, and complex far field

This example is opt-in: `RUN_GPU=False` keeps notebook execution safe on machines without CUDA. It uses one strict circle solve with 21 frequency bins.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/fdtdmesh').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
import matplotlib.pyplot as plt
import fdtdmesh as fd
OUT = Path('artifacts/notebooks')
OUT.mkdir(parents=True, exist_ok=True)
RUN_GPU = False


In [ ]:
lam = fd.C0 / 1e9
settings = fd.SolverSettings(dft_bins=21, stop=fd.DFTConvergence(check_interval=256, stable_checks=3))
sim = fd.Simulation(fmin=0.9e9, fmax=1.1e9, solver=settings)
sim.add_circle((0.0, 0.0), 0.47*lam, name='cylinder')
sim.apply_mesh('geometry_aware', cells=(144, 144))
sim.plot_geometry(mesh=True, units='wavelength', figsize=(16,14)).savefig(OUT / '10_solver_mesh.png', dpi=120, bbox_inches='tight')
result = None
if RUN_GPU:
    result = sim.solve(progress=True)
    print('converged:', result.converged, 'bins:', len(result.frequencies))


## Convergence and polar scattering

When `RUN_GPU` is enabled, convergence is plotted for all 21 bins. The scattering plot is a polar width normalized by wavelength.

In [ ]:
if result is not None:
    result.plot_convergence().savefig(OUT / '11_convergence.png', dpi=120, bbox_inches='tight')
    mid = result.frequencies[len(result.frequencies)//2]
    fig, ax = plt.subplots(figsize=(8,7), subplot_kw={'projection':'polar'})
    result.plot_scattering(frequency=mid, normalize='wavelength', ax=ax)
    fig.savefig(OUT / '12_scattering_width.png', dpi=120, bbox_inches='tight')


## Complex far field and archive round trip

The far-field views expose polar width plus complex magnitude and phase. Archives can be loaded without starting CUDA.

In [ ]:
if result is not None:
    for component in ('magnitude', 'phase'):
        fig, ax = plt.subplots(figsize=(8,7), subplot_kw={'projection':'polar'})
        result.plot_far_field(frequency=mid, component=component, ax=ax)
        fig.savefig(OUT / f'13_far_field_{component}.png', dpi=120, bbox_inches='tight')
    archive = OUT / 'cylinder_multibin.h5'
    result.save(archive)
    loaded = fd.Result.load(archive)
    print('round trip bins:', len(loaded.frequencies), 'archive:', archive)


Compare absolute scattering width with the analytical PEC-cylinder series. This measures this example’s error; it does not qualify other geometries.

In [ ]:
if result is not None:
    import numpy as np
    from fdtdmesh.scattering import cylinder_amplitude
    f = result.frequencies[len(result.frequencies)//2]
    amplitude = cylinder_amplitude(0.47*lam, f, result.angles)
    exact_width = 4*abs(amplitude)**2 / (2*np.pi*f/fd.C0)
    actual_width = result.scattering_width[len(result.frequencies)//2]
    print('Relative cylinder width error:', np.linalg.norm(actual_width-exact_width)/np.linalg.norm(exact_width))
    fig, ax = plt.subplots(figsize=(8,7), subplot_kw={'projection':'polar'})
    ax.plot(result.angles, actual_width/lam, label='FDTD')
    ax.plot(result.angles, exact_width/lam, '--', label='Analytical')
    ax.legend()
    fig.savefig(OUT / '14_analytical_cylinder.png', dpi=120, bbox_inches='tight')
